In [1]:
!pip install -q timm opencv-python scikit-learn tqdm

In [3]:
!pip install -q kaggle
import os

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"neha_0515","key":"KGAT_bcafc5b01f2d62d2a7b9ca59ceec6c6c"}')

!chmod 600 /root/.kaggle/kaggle.json


!kaggle datasets download -d olgaparfenova/daisee

Dataset URL: https://www.kaggle.com/datasets/olgaparfenova/daisee
License(s): unknown
100% 14.3G/14.3G [02:36<00:00, 98.0MB/s]



In [4]:
!unzip -q daisee.zip

In [8]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import timm
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
class DAiSEEDataset(Dataset):
    def __init__(self, video_dir, label_file, num_frames=16):
        self.video_dir = video_dir
        self.num_frames = num_frames
        self.samples = []

        df = pd.read_csv(label_file)

        for _, row in df.iterrows():
            video_name = row["ClipID"]
            engagement = int(row["Engagement"])

            video_id = video_name.split('.')[0]
            subject_id = video_id[:6]

            video_path = os.path.join(
                video_dir,
                subject_id,
                video_id,
                video_name
            )

            if os.path.exists(video_path):
                self.samples.append((video_path, engagement))

        print(f"{label_file} → {len(self.samples)} samples loaded")

    def sample_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total == 0:
            return np.zeros((self.num_frames, 224, 224, 3))

        indices = np.linspace(0, total - 1, self.num_frames).astype(int)
        indices = set(indices)

        frames = []
        i = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if i in indices:
                frame = cv2.resize(frame, (224, 224))
                frame = frame / 255.0
                frames.append(frame)

            i += 1

        cap.release()

        while len(frames) < self.num_frames:
            frames.append(frames[-1] if frames else np.zeros((224, 224, 3)))

        return np.array(frames)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        frames = self.sample_frames(path)
        frames = torch.tensor(frames).permute(0, 3, 1, 2).float()

        return frames, torch.tensor(label)

In [10]:
base_path = "/content/DAiSEE"

train_dataset = DAiSEEDataset(
    os.path.join(base_path, "DataSet/Train"),
    os.path.join(base_path, "Labels/TrainLabels.csv")
)

val_dataset = DAiSEEDataset(
    os.path.join(base_path, "DataSet/Validation"),
    os.path.join(base_path, "Labels/ValidationLabels.csv")
)

test_dataset = DAiSEEDataset(
    os.path.join(base_path, "DataSet/Test"),
    os.path.join(base_path, "Labels/TestLabels.csv")
)

/content/DAiSEE/Labels/TrainLabels.csv → 5358 samples loaded
/content/DAiSEE/Labels/ValidationLabels.csv → 1429 samples loaded
/content/DAiSEE/Labels/TestLabels.csv → 1784 samples loaded


In [11]:
class VibedNetViT(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, 4)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)
        x = self.cnn(x)
        x = x.view(B, T, -1)

        x = x.mean(dim=1)  # temporal pooling
        x = self.fc(x)

        return x

In [12]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, num_workers=2)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VibedNetViT().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [14]:
def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for frames, labels in loader:
            frames, labels = frames.to(device), labels.to(device)

            outputs = model(frames)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return acc, prec, rec, f1

In [15]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for frames, labels in tqdm(train_loader):
        frames, labels = frames.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val Precision: {val_prec:.4f}")
    print(f"Val Recall: {val_rec:.4f}")
    print(f"Val F1: {val_f1:.4f}")

100%|██████████| 1340/1340 [21:36<00:00,  1.03it/s]



Epoch 1
Loss: 1192.4159
Val Accuracy: 0.5689
Val Precision: 0.1422
Val Recall: 0.2500
Val F1: 0.1813


100%|██████████| 1340/1340 [21:05<00:00,  1.06it/s]



Epoch 2
Loss: 1168.7417
Val Accuracy: 0.5689
Val Precision: 0.1422
Val Recall: 0.2500
Val F1: 0.1813


100%|██████████| 1340/1340 [21:04<00:00,  1.06it/s]



Epoch 3
Loss: 1164.8115
Val Accuracy: 0.5689
Val Precision: 0.1422
Val Recall: 0.2500
Val F1: 0.1813


100%|██████████| 1340/1340 [21:23<00:00,  1.04it/s]



Epoch 4
Loss: 1161.7953
Val Accuracy: 0.3156
Val Precision: 0.2106
Val Recall: 0.2483
Val F1: 0.1241


100%|██████████| 1340/1340 [21:31<00:00,  1.04it/s]



Epoch 5
Loss: 1158.7665
Val Accuracy: 0.5290
Val Precision: 0.2519
Val Recall: 0.2861
Val F1: 0.2678


In [16]:
test_acc, test_prec, test_rec, test_f1 = evaluate(model, test_loader)

print("\n===== FINAL TEST RESULTS =====")
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_rec:.4f}")
print(f"F1 Score: {test_f1:.4f}")


===== FINAL TEST RESULTS =====
Accuracy: 0.5353
Precision: 0.2769
Recall: 0.2751
F1 Score: 0.2461
